# 01 — EDA exploratorio (Fase 3)

**TP Churn E-commerce** · Dataset: `data/raw/ecommerce.csv` (solo lectura)

**Pregunta de negocio** (Fase 2): ¿Qué clientes tienen mayor probabilidad de irse y qué señales de comportamiento reciente explican ese riesgo?

Este notebook explora la base **sin limpiar ni modelar**. El tratamiento de nulos va en Fase 4; las hipótesis formales en Fase 5.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4)

DATA_PATH = Path("../data/raw/ecommerce.csv")
TARGET = "Churn"
ID_COL = "CustomerID"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 1. Panorama general

In [ ]:
df.info()

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols + [ID_COL, TARGET]]

print("Categóricas:", cat_cols)
print("Numéricas (features):", num_cols)

## 2. Variable objetivo — desbalanceo de clases

In [ ]:
churn_counts = df[TARGET].value_counts().sort_index()
churn_rate = df[TARGET].mean()

print(churn_counts)
print(f"\nTasa de churn: {churn_rate:.2%}")
print("→ Clase minoritaria: no usar accuracy como métrica principal (Fase 9).")

fig, ax = plt.subplots()
sns.countplot(data=df, x=TARGET, ax=ax)
ax.set_title("Distribución de Churn")
ax.set_xticklabels(["Activo (0)", "Churn (1)"])
plt.tight_layout()
plt.show()

## 3. Valores faltantes (7 columnas — Fase 4)

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"nulos": missing, "%": missing_pct})
missing_df[missing_df["nulos"] > 0]

In [ ]:
cols_with_nulls = missing_df[missing_df["nulos"] > 0].index.tolist()

fig, ax = plt.subplots(figsize=(8, 4))
missing_df.loc[cols_with_nulls, "%"].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("% de nulos por columna")
ax.set_xlabel("%")
plt.tight_layout()
plt.show()

# ¿Los nulos se concentran en churners?
null_by_churn = df.groupby(TARGET)[cols_with_nulls].apply(lambda g: g.isnull().mean())
print("Proporción de nulos por clase (churn vs activo):")
display((null_by_churn * 100).round(2))

## 4. Features numéricas — resumen y distribuciones

In [ ]:
df[num_cols].describe().T

In [ ]:
key_nums = ["Tenure", "SatisfactionScore", "DaySinceLastOrder", "CashbackAmount", "Complain"]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, col in zip(axes, key_nums):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, element="step", ax=ax, stat="density", common_norm=False)
    ax.set_title(col)
axes[-1].axis("off")
plt.suptitle("Distribución por Churn — variables clave", y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlación con Churn

In [ ]:
corr_target = df[num_cols + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
corr_sorted = corr_target.sort_values(key=abs, ascending=False)

print("Top correlaciones con Churn:")
display(corr_sorted.head(10).to_frame("corr"))

fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(
    df[num_cols + [TARGET]].corr(numeric_only=True),
    cmap="RdBu_r", center=0, ax=ax, annot=False,
)
ax.set_title("Matriz de correlación (numéricas + Churn)")
plt.tight_layout()
plt.show()

## 6. Variables categóricas — tasa de churn por grupo

In [ ]:
def churn_rate_by(col):
    out = df.groupby(col)[TARGET].agg(["mean", "count"]).sort_values("mean", ascending=False)
    out.columns = ["churn_rate", "n"]
    return out

for col in cat_cols:
    print(f"\n=== {col} ===")
    display(churn_rate_by(col))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_cats = ["PreferredLoginDevice", "PreferedOrderCat", "MaritalStatus", "PreferredPaymentMode"]
for ax, col in zip(axes.flatten(), plot_cats):
    rates = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    rates.plot(kind="barh", ax=ax, color="coral")
    ax.set_title(f"Churn rate — {col}")
    ax.set_xlabel("P(churn)")
plt.tight_layout()
plt.show()

## 7. Deep dive — señales sospechosas (Fase 2)

In [ ]:
# Complain
complain_tbl = df.groupby("Complain")[TARGET].agg(["mean", "count"])
complain_tbl.index = complain_tbl.index.map({0: "Sin queja", 1: "Con queja"})
print("Complain vs Churn:")
display(complain_tbl)

fig, ax = plt.subplots()
sns.barplot(data=df, x="Complain", y=TARGET, estimator="mean", errorbar=None, ax=ax)
ax.set_xticklabels(["Sin queja", "Con queja"])
ax.set_ylabel("Tasa de churn")
ax.set_title("Complain — casi 3× más churn con queja")
plt.tight_layout()
plt.show()

In [ ]:
# Tenure (antigüedad)
df["Tenure_bin"] = pd.cut(
    df["Tenure"],
    bins=[0, 6, 12, 24, 61],
    labels=["0-5 meses", "6-11", "12-23", "24+"],
    right=False,
)
tenure_tbl = df.groupby("Tenure_bin", observed=True)[TARGET].agg(["mean", "count"])
print("Tenure vs Churn:")
display(tenure_tbl)

sns.barplot(data=df, x="Tenure_bin", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por antigüedad")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

In [ ]:
# SatisfactionScore — ojo: relación no lineal / contra-intuitiva
sat_tbl = df.groupby("SatisfactionScore")[TARGET].agg(["mean", "count"])
print("SatisfactionScore vs Churn:")
display(sat_tbl)

sns.barplot(data=df, x="SatisfactionScore", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por puntaje de satisfacción")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

In [ ]:
# DaySinceLastOrder
df["Days_bin"] = pd.cut(
    df["DaySinceLastOrder"],
    bins=[-1, 7, 30, 90, 999],
    labels=["0-7 días", "8-30 días", "31-90 días", "90+ días"],
)
days_tbl = df.groupby("Days_bin", observed=True)[TARGET].agg(["mean", "count"])
print("DaySinceLastOrder vs Churn:")
display(days_tbl)

sns.barplot(data=df, x="Days_bin", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por días desde último pedido")
plt.ylabel("Tasa de churn")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Hallazgos preliminares (input para Fase 4–5)

| Hallazgo | Detalle | Implicación |
|----------|---------|-------------|
| **Desbalanceo** | 16,84% churn | Métrica ≠ accuracy |
| **Nulos** | 7 columnas, ~4,5–5,5% cada una | Tratamiento en Fase 4; revisar si churners tienen más nulos |
| **Tenure** | Clientes 0–5 meses: ~35% churn; 24+ meses: ~0% | Señal fuerte; clientes nuevos = foco de retención |
| **Complain** | 31,7% churn con queja vs 10,9% sin queja | Alerta temprana; validar leakage en Fase 6 |
| **SatisfactionScore** | Score 5 tiene *más* churn que score 1 | No asumir relación lineal; explorar interacciones en Fase 5 |
| **DaySinceLastOrder** | Mayor churn en 0–7 días (19,2%) que en 8–30 (9,4%) | Proxy de inactividad distinto al esperado — investigar definición de churn |
| **Categóricas** | Single, Mobile Phone, COD/E-wallet lideran churn | Segmentos para campañas |

**Próximo paso**: Fase 4 — decisión de imputación por columna. Ver `reports/03_eda_exploratorio.md`.